# Modelo Pytorch com Adam

## Carregar DF

In [1]:
import pandas as pd
df = pd.read_csv('df_exportado.csv')

In [3]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold, train_test_split
from sklearn.feature_extraction.text import CountVectorizer
import itertools

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"A usar: {device}")

# ---------------------------------------------------
# 1. LABELS E SPLIT
# ---------------------------------------------------
label_map = {'Human':0,'OpenAI':1,'Google':2,'Meta':3,'Anthropic':4}
y = df['Label'].map(label_map).values
num_classes = 5

train_texts, test_texts, y_train, y_test = train_test_split(
    df['Text'], y, test_size=0.2, random_state=42, stratify=y
)

# ---------------------------------------------------
# 2. VETORIZAÇÃO OTIMIZADA (BIGRAMS)
# ---------------------------------------------------
max_words = 5000

vectorizer = CountVectorizer(
    max_features=max_words,
    ngram_range=(1, 2),
    stop_words="english"
)

print("A criar vetores e a transferir para tensores...")
X_train_np = vectorizer.fit_transform(train_texts).toarray()
y_train_np = np.array(y_train)

# Transformar para tensores do PyTorch de uma vez só!
X_train_tensor = torch.tensor(X_train_np, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_np, dtype=torch.long)

# ---------------------------------------------------
# 3. MODELO (Com Batch Normalization para treinar mais rápido)
# ---------------------------------------------------
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

# ---------------------------------------------------
# 4. FUNÇÕES DE TREINO E AVALIAÇÃO OTIMIZADAS
# ---------------------------------------------------
def train_model(model, loader, criterion, optimizer):
    model.train()
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
    return correct / total

# ---------------------------------------------------
# 5. GRID SEARCH (Amostra Reduzida para Rapidez)
# ---------------------------------------------------
param_grid = {
    'learning_rate': [0.001, 0.0005], # Adam prefere LRs mais baixos
    'batch_size': [128, 256],         # Batches maiores no PyTorch = muito mais rápido
    'epochs': [5]
}

keys, values = zip(*param_grid.items())
configs = [dict(zip(keys, v)) for v in itertools.product(*values)]

# OTIMIZAÇÃO: Usar apenas 20.000 textos para descobrir os melhores parâmetros
tamanho_amostra = min(20000, len(X_train_tensor))
indices_amostra = torch.randperm(len(X_train_tensor))[:tamanho_amostra]
X_grid = X_train_tensor[indices_amostra]
y_grid = y_train_tensor[indices_amostra]

kf = KFold(n_splits=3, shuffle=True, random_state=42)

melhor_acc = 0
melhores_params = None

pesos = torch.tensor([1.0, 1.0, 1.0, 1.0, 1.5]).to(device)

print("\nGrid Search Rápido...")
for config in configs:
    fold_acc = []
    # K-Fold converte indices do numpy para pytorch
    for train_idx, val_idx in kf.split(X_grid.numpy()):
        
        # TensorDataset é muito mais rápido que criar uma classe Dataset customizada
        train_dataset = TensorDataset(X_grid[train_idx], y_grid[train_idx])
        val_dataset = TensorDataset(X_grid[val_idx], y_grid[val_idx])

        # num_workers=0 no Windows para evitar crashes. Se usares Linux/Mac, mete num_workers=2
        train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=config['batch_size'])

        model = MLP(max_words, num_classes).to(device)
        criterion = nn.CrossEntropyLoss(weight=pesos)
        optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])

        for epoch in range(config['epochs']):
            train_model(model, train_loader, criterion, optimizer)

        acc = evaluate(model, val_loader)
        fold_acc.append(acc)

    media = np.mean(fold_acc)
    print(f"{config} -> Accuracy: {media:.4f}")

    if media > melhor_acc:
        melhor_acc = media
        melhores_params = config

print("\nMelhores parâmetros:", melhores_params)

# ---------------------------------------------------
# 6. TREINO FINAL
# ---------------------------------------------------
X_train_final, X_val_final, y_train_final, y_val_final = train_test_split(
    X_train_tensor.numpy(), y_train_tensor.numpy(), test_size=0.15, stratify=y_train_tensor.numpy(), random_state=42
)

# Reconversão para Tensores
X_train_final = torch.tensor(X_train_final, dtype=torch.float32)
y_train_final = torch.tensor(y_train_final, dtype=torch.long)
X_val_final = torch.tensor(X_val_final, dtype=torch.float32)
y_val_final = torch.tensor(y_val_final, dtype=torch.long)

train_dataset = TensorDataset(X_train_final, y_train_final)
val_dataset = TensorDataset(X_val_final, y_val_final)

train_loader = DataLoader(train_dataset, batch_size=melhores_params['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=melhores_params['batch_size'])

model_final = MLP(max_words, num_classes).to(device)
criterion = nn.CrossEntropyLoss(weight=pesos)
optimizer = torch.optim.Adam(model_final.parameters(), lr=melhores_params['learning_rate'])

print("\nTreino final...")
for epoch in range(15):
    train_model(model_final, train_loader, criterion, optimizer)
    
    train_acc = evaluate(model_final, train_loader)
    val_acc = evaluate(model_final, val_loader)
    print(f"Epoch {epoch+1:02d} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")

print("\nTreino Concluído!")

A usar: cpu
A criar vetores e a transferir para tensores...

Grid Search Rápido...
{'learning_rate': 0.001, 'batch_size': 128, 'epochs': 5} -> Accuracy: 0.8899
{'learning_rate': 0.001, 'batch_size': 256, 'epochs': 5} -> Accuracy: 0.8965
{'learning_rate': 0.0005, 'batch_size': 128, 'epochs': 5} -> Accuracy: 0.8961
{'learning_rate': 0.0005, 'batch_size': 256, 'epochs': 5} -> Accuracy: 0.9007

Melhores parâmetros: {'learning_rate': 0.0005, 'batch_size': 256, 'epochs': 5}

Treino final...
Epoch 01 | Train: 0.9696 | Val: 0.9453
Epoch 02 | Train: 0.9848 | Val: 0.9529
Epoch 03 | Train: 0.9915 | Val: 0.9562
Epoch 04 | Train: 0.9956 | Val: 0.9599
Epoch 05 | Train: 0.9975 | Val: 0.9589
Epoch 06 | Train: 0.9984 | Val: 0.9601
Epoch 07 | Train: 0.9989 | Val: 0.9607
Epoch 08 | Train: 0.9991 | Val: 0.9610
Epoch 09 | Train: 0.9996 | Val: 0.9611
Epoch 10 | Train: 0.9996 | Val: 0.9620
Epoch 11 | Train: 0.9997 | Val: 0.9612
Epoch 12 | Train: 0.9997 | Val: 0.9612
Epoch 13 | Train: 0.9998 | Val: 0.9614
Epo

In [6]:
import pickle
# ==========================================
# 1. GUARDAR O VECTORIZER (Vocabulário)
# ==========================================
with open('vectorizer_pytorch.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
print("Vectorizer guardado com sucesso!")

# ==========================================
# 2. GUARDAR O MODELO PYTORCH
# ==========================================
caminho_modelo = 'pesos_pytorch.pth'

# Passar o modelo para CPU antes de guardar (para garantir que abre em qualquer PC)
model_final = model_final.to('cpu') 
torch.save(model_final.state_dict(), caminho_modelo)

Vectorizer guardado com sucesso!
